# 02 — LLM Insight Generation

Validate the LLM insight generation pipeline end-to-end:
- Load analytics from S3 (written by `01_umami_ingest`)
- Call `generate_insights()` — the same production function the agent runs daily
- Works with any provider: set `LLM_PROVIDER=ionos` or `LLM_PROVIDER=mistral` in `.env`

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os
import logging

# Surface log output from production code
logging.basicConfig(level=logging.INFO, format='%(name)s %(levelname)s: %(message)s')

load_dotenv('../.env')

# Set provider here — this is picked up by generate_insights() via os.environ
os.environ['LLM_PROVIDER'] = os.environ.get('LLM_PROVIDER', 'ionos')  # override if needed, e.g. 'mistral'

from agent.llm_client import PROVIDERS
provider = os.environ['LLM_PROVIDER']
config = PROVIDERS.get(provider)
api_key_set = bool(os.environ.get(config.api_key_env)) if config else False

print(f'Active provider : {provider}')
print(f'API key loaded  : {"yes" if api_key_set else f"NO — set {config.api_key_env if config else \"LLM_PROVIDER\"} in .env"}')

## 1. Load Analytics Data

Load the insights from `01_umami_ingest` notebook output.

In [7]:
from agent.storage import S3Storage

store = S3Storage(
    bucket=os.getenv('S3_BUCKET'),
    prefix=os.getenv('S3_STATE_PREFIX', 'growth-agent/'),
    access_key=os.getenv('SCW_ACCESS_KEY'),
    secret_key=os.getenv('SCW_SECRET_KEY'),
)
insights_raw = store.read('insights.json')

if insights_raw is None:
    print('ERROR: Run 01_umami_ingest.ipynb first to generate insights.json')
else:
    print(f'Loaded insights : {insights_raw["website_analytics"]["visitors"]} visitors')
    print(f'Top pages       : {len(insights_raw["website_analytics"]["top_pages"])}')

Loaded insights : 43 visitors
Top pages       : 10


## 2. Run Production Insight Generation

Calls the same `generate_insights()` function the agent uses in production.
The LLM provider and model are picked up from `LLM_PROVIDER` / `LLM_MODEL` env vars.

In [ ]:
from agent.nodes.insights import generate_insights

# Raises on failure — check provider and API key above if this fails
analysis = generate_insights(store)

print(f'Provider        : {provider}')
print(f'Top topics      : {analysis.top_topics}')
print(f'Pages for social: {len(analysis.best_pages_for_social)}')

## 4. Inspect Structured Output

In [ ]:
# analysis is already a typed LLMAnalysis — no JSON parsing needed
print('=== Parsed Analysis ===')
print(f'\nTop topics: {analysis.top_topics}')
print(f'\nTraffic sources: {analysis.traffic_sources}')
print(f'\nBest pages for social:')
for page in analysis.best_pages_for_social:
    print(f'  - {page.url} — {page.reason}')
print(f'\nContent gaps: {analysis.content_gaps}')
print(f'\nGrowth opportunities:')
for opp in analysis.growth_opportunities:
    print(f'  - {opp}')

## 5. Verify S3 Persistence

In [ ]:
# generate_insights() already saved insights.json and llm_analysis.json to S3 — just verify
saved = store.read('llm_analysis.json')
print(f'llm_analysis.json saved: {saved is not None}')
print(f'Growth opportunities: {len(saved.get("growth_opportunities", []))}')

In [ ]:
print('Done — LLM insight generation validated.')